# Build/update canonical private Kaggle core — top controls

Jalankan **Runtime → Run all** di Colab akun yang dapat mengakses folder Drive `Coffee_Bean_Detection`. Notebook ini mengikuti protokol canonical yang sebelumnya dipakai AF2 spectral: satu private Kaggle Dataset bernama `faruq-v3-experiment-core-v1`, arsip development disimpan opaque sebagai `.tar.bin`, seluruh checkpoint diverifikasi ukuran/SHA/seed dan di-load-test dengan Ultralytics 8.4.96 sebelum upload.

Hasil training tidak dicampurkan ke core dataset. Resume hanya berasal dari output **Kaggle Saved Version** dengan kontrak arm/config/seed/checkpoint yang identik. Test tidak dimuat atau diakses.

Buat dua Colab secrets dan aktifkan notebook access: `KAGGLE_USERNAME` dan `KAGGLE_API_TOKEN`.


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=True)

import importlib, json, os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'codex/top-controls-multiseed-confirmation'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
for attempt in range(3):
    result = subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', BRANCH,
        'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)
    ])
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 2:
        raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.96', 'kaggle'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
importlib.invalidate_caches()
os.chdir(REPO)
print('REPO COMMIT:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())
print('ULTRALYTICS:', __import__('ultralytics').__version__)


In [ ]:
from coffee_detector.drive_project import resolve_drive_project_root
from coffee_detector.experiments.prepare_faruq_v3_kaggle import (
    ARCHIVE_BYTES, ARCHIVE_NAME, ARCHIVE_SHA256, EXPECTED_ANNOTATIONS, EXPECTED_IMAGES
)
from coffee_detector.experiments.prepare_top_controls_kaggle import (
    CORE_MANIFEST_FORMAT, build_top_controls_canonical_kaggle_core
)

PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-stb-paired-confirmation-v1/STB1/STB1_seed123/weights/best.pt',
    'experiments/faruq-v3-stb-paired-confirmation-v1/STB1/STB1_seed2026/weights/best.pt',
    'experiments/faruq-v3-af2-igem-paired-confirmation-v1/AF2/AF2_seed123/weights/best.pt',
    'experiments/faruq-v3-af2-igem-paired-confirmation-v1/AF2/AF2_seed2026/weights/best.pt',
))
BUNDLE = Path('/content/faruq-v3-experiment-core-v1')
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
core = build_top_controls_canonical_kaggle_core(PROJECT_ROOT, BUNDLE)

assert core['format'] == CORE_MANIFEST_FORMAT
assert core['canonical_dataset_slug'] == 'faruq-v3-experiment-core-v1'
assert core['archive'] == {
    'name': ARCHIVE_NAME, 'bytes': ARCHIVE_BYTES, 'sha256': ARCHIVE_SHA256, 'opaque': True
}
assert core['completed_training_runs_included'] is False
assert core['resume_source'] == 'kaggle_saved_version_only'
assert core['test_images_included'] is False
assert not list(BUNDLE.rglob('run_contract.json')), 'Core tidak boleh berisi hasil training/resume.'

print('PROJECT:', PROJECT_ROOT)
print('BUNDLE :', BUNDLE)
print('ARCHIVE:', ARCHIVE_NAME, ARCHIVE_BYTES, ARCHIVE_SHA256)
print('KONTRAK DATASET SAAT PREFLIGHT KAGGLE:')
print('  images     :', EXPECTED_IMAGES)
print('  annotations:', EXPECTED_ANNOTATIONS)
print('  classes    : 0..20 pada train dan val')
print('CHECKPOINT LOAD-TEST:')
for name, proof in core['checkpoint_validation'].items():
    print(f"  {name}: seed={proof['seed']} nc={proof['nc']} params={proof['parameters']} bytes={proof['bytes']} sha256={proof['sha256']} loadable={proof['loadable_by_ultralytics']}")
print('BUILD CANONICAL CORE: PASS')


In [ ]:
username = userdata.get('KAGGLE_USERNAME')
token = userdata.get('KAGGLE_API_TOKEN')
assert username and token, 'Aktifkan secret KAGGLE_USERNAME dan KAGGLE_API_TOKEN.'
username = username.strip()
assert username and '/' not in username, 'KAGGLE_USERNAME tidak valid.'
os.environ['KAGGLE_USERNAME'] = username
os.environ['KAGGLE_API_TOKEN'] = token
os.environ['KAGGLE_KEY'] = token

dataset_id = f'{username}/faruq-v3-experiment-core-v1'
metadata = {
    'title': 'Faruq V3 Experiment Core V1',
    'id': dataset_id,
    'licenses': [{'name': 'other'}],
    'isPrivate': True,
}
(BUNDLE / 'dataset-metadata.json').write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')
message = 'Canonical Faruq-v3 opaque archive plus top-control seed-matched checkpoints and SHA contracts'
version = subprocess.run(
    ['kaggle', 'datasets', 'version', '-p', str(BUNDLE), '-m', message, '--keep-tabular'],
    text=True, capture_output=True
)
if version.returncode == 0:
    print(version.stdout)
    action = 'UPDATED'
else:
    combined = (version.stdout + '\n' + version.stderr).lower()
    missing = any(text in combined for text in ('not found', '404', 'does not exist'))
    if not missing:
        print(version.stdout)
        print(version.stderr)
        raise RuntimeError(f'Kaggle dataset version gagal: returncode={version.returncode}')
    print('Dataset belum ada pada akun baru; membuat private dataset canonical...', flush=True)
    subprocess.run(
        ['kaggle', 'datasets', 'create', '-p', str(BUNDLE), '--keep-tabular'],
        check=True
    )
    action = 'CREATED'

url = f'https://www.kaggle.com/datasets/{dataset_id}'
print('UPLOAD:', action)
print('DATASET:', dataset_id)
print('URL:', url)
print('PRIVATE: True | TEST: False')


In [ ]:
required = (
    'faruq-development-v3-grouped.tar.bin',
    'top_controls_kaggle_manifest.json',
    'top_controls_canonical_core_manifest.json',
    'STB1_seed123_best.pt', 'STB1_seed2026_best.pt',
    'AF2_seed123_best.pt', 'AF2_seed2026_best.pt',
)
listing = ''
for attempt in range(12):
    check = subprocess.run(
        ['kaggle', 'datasets', 'files', dataset_id, '--page-size', '100', '--csv'],
        text=True, capture_output=True
    )
    listing = check.stdout
    if check.returncode == 0 and all(name in listing for name in required):
        break
    print(f'Menunggu indeks Kaggle: {attempt + 1}/12', flush=True)
    time.sleep(10)
else:
    print(listing)
    print(check.stderr)
    raise RuntimeError('Upload diterima tetapi file canonical belum lengkap di indeks Kaggle.')

print('VERIFIKASI KAGGLE: PASS')
print('Pasang dataset ini ke notebook sequential:', dataset_id)
print('Jika training terputus, pasang output Saved Version sebelumnya sebagai Input tambahan.')
